# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [1952, 1957, 1962, 1967, 1972, 1977, 1982, 1987, 1992, 1997, 2002, 2007]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    425.9
std      17.3     12.9  1.061579e+08     9857.5    248.3
min    1952.0     23.6  6.001100e+04      241.2      4.0


## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [3]:
# Compare European countries in 2002 and 2007; connecting lines show direction.
europe = df.loc[
    df["continent"].eq("Europe") & df["year"].isin([2002, 2007]),
    ["country", "year", "gdpPercap", "lifeExp"],
].sort_values(["country", "year"]).copy()
europe["year_label"] = europe["year"].astype(str)

fig = px.line(
    europe,
    x="gdpPercap",
    y="lifeExp",
    line_group="country",
    hover_name="country",
    color_discrete_sequence=["#B8B8B8"],
)
fig.update_traces(showlegend=False, line={"width": 1})

year_points = px.scatter(
    europe,
    x="gdpPercap",
    y="lifeExp",
    color="year_label",
    hover_name="country",
    hover_data={"year_label": False, "year": True, "gdpPercap": ":,.0f", "lifeExp": ":.1f"},
    category_orders={"year_label": ["2002", "2007"]},
    color_discrete_map={"2002": "#2878B5", "2007": "#D55E00"},
)
for trace in year_points.data:
    fig.add_trace(trace)

fig.update_layout(
    title="European countries gained income and life expectancy from 2002 to 2007",
    title_x=0.5,
    xaxis_title="GDP per capita (log scale)",
    yaxis_title="Life expectancy (years)",
    legend_title="Year",
)
fig.update_xaxes(type="log")
fig.show()


## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [4]:
# Compare Asia with all other continents in grey for 2007.
df_2007 = df.loc[df["year"].eq(2007)].copy()
highlighted_continent = "Asia"
df_2007["highlight_group"] = df_2007["continent"].where(
    df_2007["continent"].eq(highlighted_continent), "Other continents"
)

fig = px.scatter(
    df_2007,
    x="gdpPercap",
    y="lifeExp",
    size="pop",
    size_max=45,
    color="highlight_group",
    hover_name="country",
    hover_data={"continent": True, "gdpPercap": ":,.0f", "lifeExp": ":.1f", "pop": ":,.0f"},
    category_orders={"highlight_group": ["Other continents", highlighted_continent]},
    color_discrete_map={
        "Other continents": "#C9CDD2",
        highlighted_continent: "#2878B5",
    },
)

china = df_2007.loc[df_2007["country"].eq("China")].iloc[0]
fig.add_annotation(
    x=china["gdpPercap"],
    y=china["lifeExp"],
    text=f"China: life expectancy was about {china['lifeExp']:.0f} years",
    showarrow=True,
    arrowhead=2,
    ax=65,
    ay=-55,
    bgcolor="white",
    bordercolor="#2878B5",
    borderpad=4,
)
fig.update_layout(
    title="Asia spans a wide range of incomes, while life expectancy is high across much of the region (2007)",
    title_x=0.5,
    xaxis_title="GDP per capita (log scale)",
    yaxis_title="Life expectancy (years)",
    legend_title="Continent",
)
fig.update_xaxes(type="log")
fig.show()
